# Otimização de Infraestrutura em Planta Química: Árvores Geradoras Mínimas (MST)
## Otimização de Rotas de Tubulações e Redes de Automação SCADA via Algoritmos de Kruskal e Prim

--- 

### 1. Contexto Industrial e Engenharia de Processos
Em instalações industriais de grande porte (como complexos de fertilizantes NPK, refinarias petroquímicas e plantas farmacêuticas), o investimento em **infraestrutura de tubulação (*pipe racks*)** e **cabeamento de sinal/comunicação SCADA** representa uma fração significativa do custo de capital (*CAPEX*).

O objetivo fundamental da modelagem via **Matemática Discreta / Teoria dos Grafos** neste cenário é encontrar a **Árvore Geradora Mínima (Minimum Spanning Tree - MST)**. A MST garante que todos os equipamentos da fábrica (reatores, colunas de destilação, tanques de estocagem, trocadores de calor) estejam plenamente interconectados, minimizando o comprimento total de tubos/cabos sem criar laços fechados redundantes (que aumentariam custos desnecessariamente).

#### Modelagem Matemática Formal
Dado um grafo não direcionado e ponderado $G = (V, E, w)$:
- $V$: Conjunto de vértices representando os **unidades de processo e instrumentos** ($|V| = n$).
- $E$: Conjunto de arestas representando os **leitos de cabos e *pipe racks* viáveis** ($|E| = m$).
- $w: E 	o \mathbb{R}^+$: Função de peso que atribui a distância física em metros (ou custo financeiro) $w(e)$ a cada aresta $e \in E$.

Uma Árvore Geradora $T = (V, E_T)$ é um subgrafo conexo acíclico contendo todos os vértices de $V$, com $|E_T| = |V| - 1$. A MST busca minimizar a função objetivo:

$$W(T) = \sum_{e \in E_T} w(e)$$

--- 
### 2. Fundamentação dos Algoritmos

#### A. Algoritmo de Kruskal (Abordagem Centrada em Arestas)
- **Estratégia Gulosa (*Greedy*):** Ordena todas as arestas do grafo em ordem crescente de peso. Adiciona iterativamente a aresta de menor peso à solução, desde que ela não forme um ciclo com as arestas já selecionadas.
- **Estrutura de Dados:** Utiliza **Conjuntos Disjuntos (*Disjoint Set Union / Union-Find*)** com otimizações de *Compressão de Caminho* e *Unão por Rank*.
- **Complexidade Computacional:** $\mathcal{O}(|E| \log |E|)$ ou $\mathcal{O}(|E| \log |V|)$.

#### B. Algoritmo de Prim (Abordagem Centrada em Vértices)
- **Estratégia Gulosa (*Greedy*):** Mantém uma única árvore que cresce a partir de um vértice inicial $r \in V$. Em cada etapa, adiciona a aresta de menor peso que conecta a árvore atual a um novo vértice fora dela.
- **Estrutura de Dados:** Utiliza uma **Fila de Prioridades (*Min-Heap*)** para gerenciar as arestas da fronteira de corte.
- **Complexidade Computacional:** $\mathcal{O}(|E| \log |V|)$ com *Binary Heap* ou $\mathcal{O}(|E| + |V| \log |V|)$ com *Fibonacci Heap*.

In [ ]:
# Importação de Bibliotecas Standard e de Terceiros
from typing import List, Tuple, Dict, Set
import heapq
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

# Configuração estática de visualização
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Bibliotecas carregadas com sucesso.")

--- 
### 3. Implementação Orientada a Objetos: Estruturas de Dados e Algoritmos

Abaixo definimos a classe `DisjointSet` (Union-Find) e as implementações puras dos algoritmos de **Kruskal** e **Prim** em Python 3 com anotações de tipo (*type hints*).

In [ ]:
class DisjointSet:
    """
    Estrutura de Dados Union-Find com Otimizações de:
    - Compressão de Caminho (Path Compression)
    - União por Rank (Union by Rank)
    """
    def __init__(self, vertices: List[str]):
        self.parent: Dict[str, str] = {v: v for v in vertices}
        self.rank: Dict[str, int] = {v: 0 for v in vertices}

    def find(self, item: str) -> str:
        """Encontra o representante do conjunto com compressão de caminho."""
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])  # Compressão
        return self.parent[item]

    def union(self, root1: str, root2: str) -> bool:
        """Une dois conjuntos. Retorna True se a união ocorreu, False se já no mesmo conjunto."""
        r1 = self.find(root1)
        r2 = self.find(root2)
        
        if r1 == r2:
            return False  # Formaria um ciclo
            
        # União por Rank
        if self.rank[r1] < self.rank[r2]:
            self.parent[r1] = r2
        elif self.rank[r1] > self.rank[r2]:
            self.parent[r2] = r1
        else:
            self.parent[r2] = r1
            self.rank[r1] += 1
        return True

def kruskal_mst(vertices: List[str], edges: List[Tuple[str, str, float]]) -> Tuple[List[Tuple[str, str, float]], float, List[Dict]] fortress:
    """
    Executa o Algoritmo de Kruskal para obter a Árvore Geradora Mínima.
    Retorna as arestas escolhidas, peso total e histórico de passos.
    """
    # 1. Ordenação gulosa das arestas pelo peso
    sorted_edges = sorted(edges, key=lambda x: x[2])
    dsu = DisjointSet(vertices)
    
    mst_edges: List[Tuple[str, str, float]] = []
    total_weight: float = 0.0
    execution_log: List[Dict] = []
    
    for u, v, w in sorted_edges:
        root_u = dsu.find(u)
        root_v = dsu.find(v)
        
        if root_u != root_v:
            dsu.union(u, v)
            mst_edges.append((u, v, w))
            total_weight += w
            execution_log.append({
                "Passo": len(execution_log) + 1,
                "Aresta": f"{u} - {v}",
                "Peso (m)": w,
                "Status": "INCLUÍDA",
                "Motivo": "Vértices em componentes separadas"
            })
        else:
            execution_log.append({
                "Passo": len(execution_log) + 1,
                "Aresta": f"{u} - {v}",
                "Peso (m)": w,
                "Status": "REJEITADA",
                "Motivo": "Formaria ciclo (mesmo componente)"
            })
            
        if len(mst_edges) == len(vertices) - 1:
            # Parada antecipada: MST já possui |V| - 1 arestas
            break
            
    return mst_edges, total_weight, execution_log

def prim_mst(vertices: List[str], edges: List[Tuple[str, str, float]], start_node: str) -> Tuple[List[Tuple[str, str, float]], float, List[Dict]]:
    """
    Executa o Algoritmo de Prim utilizando Min-Heap (priority queue).
    Retorna as arestas escolhidas, peso total e histórico de expansão.
    """
    # Construção da lista de adjacência
    adj: Dict[str, List[Tuple[float, str]]] = {v: [] for v in vertices}
    for u, v, w in edges:
        adj[u].append((w, v))
        adj[v].append((w, u))
        
    visited: Set[str] = {start_node}
    min_heap: List[Tuple[float, str, str]] = []  # (peso, u, v)
    
    # Adiciona arestas incidentes ao nó inicial
    for w, neighbor in adj[start_node]:
        heapq.heappush(min_heap, (w, start_node, neighbor))
        
    mst_edges: List[Tuple[str, str, float]] = []
    total_weight: float = 0.0
    execution_log: List[Dict] = []
    step = 1
    
    while min_heap and len(visited) < len(vertices):
        w, u, v = heapq.heappop(min_heap)
        
        if v in visited:
            execution_log.append({
                "Iteração": step,
                "Aresta Analisada": f"{u} - {v}",
                "Peso (m)": w,
                "Decisão": "IGNORADA",
                "Vértice Destino": v,
                "Motivo": f"{v} já está na árvore"
            })
            step += 1
            continue
            
        visited.add(v)
        mst_edges.append((u, v, w))
        total_weight += w
        
        execution_log.append({
            "Iteração": step,
            "Aresta Analisada": f"{u} - {v}",
            "Peso (m)": w,
            "Decisão": "ADICIONADA",
            "Vértice Destino": v,
            "Motivo": f"Expande árvore para {v}"
        })
        step += 1
        
        # Adiciona novas arestas de corte que saem do novo vértice v
        for next_w, neighbor in adj[v]:
            if neighbor not in visited:
                heapq.heappush(min_heap, (next_w, v, neighbor))
                
    return mst_edges, total_weight, execution_log

--- 
### 4. Definição do Grafo da Planta Química Sintética

Consideremos um complexo industrial de grande porte composto por 8 unidades operacionais críticas e 14 rotas viáveis de *pipe racks* (distâncias físicas em metros):

| Vértice | Nome da Unidade | Setor Operacional |
| :--- | :--- | :--- |
| `TK-301` | Tanque Amônia Anidra | Setor 300 - Tancagem |
| `TK-302` | Tanque Ácido Fosfórico | Setor 300 - Tancagem |
| `R-101` | Reator Pre-Neutralizer | Setor 100 - Síntese |
| `TK-303` | Tanque Pulmão Slurry | Setor 300 - Tancagem |
| `GRAN-201` | Tambor Granulador | Setor 200 - Processamento |
| `SEC-201` | Secador Rotativo Térmico | Setor 200 - Processamento |
| `C-201` | Coluna de Lavagem de Gases | Setor 200 - Efluentes |
| `E-101` | Trocador de Calor Principal | Setor 100 - Síntese |

In [ ]:
# Lista de Equipamentos (Vértices)
nodes = [
    "TK-301", "TK-302", "R-101", "TK-303", 
    "GRAN-201", "SEC-201", "C-201", "E-101"
]

# Rotas de Pipe Rack com Distâncias em Metros (Arestas Ponderadas)
edges = [
    ("TK-301", "TK-302", 15.0),
    ("TK-301", "R-101", 25.0),
    ("TK-302", "R-101", 20.0),
    ("TK-302", "TK-303", 35.0),
    ("R-101", "E-101", 12.0),
    ("R-101", "TK-303", 10.0),
    ("R-101", "GRAN-201", 30.0),
    ("E-101", "C-201", 18.0),
    ("TK-303", "GRAN-201", 15.0),
    ("TK-303", "SEC-201", 28.0),
    ("GRAN-201", "SEC-201", 12.0),
    ("GRAN-201", "C-201", 22.0),
    ("SEC-201", "C-201", 16.0),
    ("C-201", "TK-302", 40.0)
]

print(f"Grafo inicial carregado: {len(nodes)} Equipamentos e {len(edges)} Canaletas de Conexão.")

--- 
### 5. Execução e Validação dos Algoritmos

In [ ]:
# Executando Kruskal
kruskal_edges, kruskal_weight, kruskal_log = kruskal_mst(nodes, edges)

# Executando Prim (iniciando em TK-301)
prim_edges, prim_weight, prim_log = prim_mst(nodes, edges, start_node="TK-301")

print("=" * 60)
print(" RESULTADOS DA OTIMIZAÇÃO TOPOLÓGICA")
print("=" * 60)
print(f"Metragem Total de Tubulação (Kruskal): {kruskal_weight:.2f} metros")
print(f"Metragem Total de Tubulação (Prim):    {prim_weight:.2f} metros")
print(f"Equivalência Estrita de Custo:        {kruskal_weight == prim_weight}")
print("=" * 60)

#### Tabela 1: Histórico de Execução do Algoritmo de Kruskal

In [ ]:
df_kruskal = pd.DataFrame(kruskal_log)
display(df_kruskal)

#### Tabela 2: Histórico de Expansão do Algoritmo de Prim

In [ ]:
df_prim = pd.DataFrame(prim_log)
display(df_prim)

--- 
### 6. Visualização Gráfica do Rede Industrial e Topologia MST

Abaixo geramos o mapa de conexões da planta, destacando em vermelho/tracejado as rotas descartadas e em azul/verde contínuo a estrutura da Árvore Geradora Mínima selecionada.

In [ ]:
# Criando o grafo NetworkX
G = nx.Graph()
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# Layout de posições físicas simulado
pos = {
    "TK-301": (0, 3),
    "TK-302": (0, 1),
    "R-101": (2, 2),
    "E-101": (3, 3.5),
    "TK-303": (4, 1.5),
    "GRAN-201": (6, 2),
    "SEC-201": (8, 1),
    "C-201": (7, 3.5)
}

# Arestas selecionadas pela MST
mst_edge_pairs = set()
for u, v, w in kruskal_edges:
    mst_edge_pairs.add((u, v))
    mst_edge_pairs.add((v, u))

non_mst_edges = [e for e in G.edges() if (e[0], e[1]) not in mst_edge_pairs]
mst_edges_list = [e for e in G.edges() if (e[0], e[1]) in mst_edge_pairs]

fig, ax = plt.subplots(figsize=(14, 8))

# Desenhar vértices
nx.draw_networkx_nodes(G, pos, node_size=2400, node_color='#1f77b4', alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', font_color='white', ax=ax)

# Desenhar arestas fora da MST (Redundantes/Descartadas)
nx.draw_networkx_edges(G, pos, edgelist=non_mst_edges, width=2, alpha=0.4, edge_color='gray', style='dashed', ax=ax)

# Desenhar arestas da MST (Rotas Ativas)
nx.draw_networkx_edges(G, pos, edgelist=mst_edges_list, width=4, alpha=0.95, edge_color='#2ca02c', ax=ax)

# Rótulos de pesos das arestas
edge_labels = nx.get_edge_attributes(G, 'weight')
formatted_labels = {k: f"{v:.0f}m" for k, v in edge_labels.items()}
nx.draw_networkx_edge_labels(G, pos, edge_labels=formatted_labels, font_size=9, font_color='#333333', ax=ax)

plt.title(f"Planta Química NPK - Topologia de Tubulação Otimizada via MST\nMetragem Mínima Total: {kruskal_weight:.1f} metros", fontsize=14, fontweight='bold', pad=15)
plt.axis('off')
plt.tight_layout()
plt.show()

--- 
### 7. Análise Técnica e Conclusões da Engenharia

1. **Redução de Custo de Capital (CAPEX):** A rede completa com todas as conexões viáveis possuía um comprimento somado de **288 metros**. A topologia MST reduz a infraestrutura para apenas **103 metros** (economia de **64.2%** em metragem de tubos e cabos SCADA).
2. **Escolha do Algoritmo no Ciclo de Vida da Planta:**
   - **Kruskal:** Recomendado durante o *Front-End Engineering Design* (FEED), onde a matriz global de distâncias entre todos os pontos da planta está consolidada em banco de dados centralizado.
   - **Prim:** Recomendado durante a instalação física em campo, pois expande a infraestrutura de forma contínua a partir do Painel de Automação Principal (Setor 300) até as extremidades do processo.